In [6]:
import os
import tensorflow as tf
import numpy as np

# 1. 경로 및 설정
test_dir = 'test_images'
resnet_model_path = 'resnet_room_clean_model.keras' 
xception_classifier_path = 'room_model.h5' # 이름 변경 (캐글 분류기)

resnet_img_size = (256, 256) 
xception_img_size = (299, 299)
class_names = ['clean', 'dirty']

# 2. 모델 로드
print("모델을 불러오는 중입니다...")
resnet_model = tf.keras.models.load_model(resnet_model_path)
xception_classifier = tf.keras.models.load_model(xception_classifier_path)

# ⭐ [추가된 부분] Xception 베이스 모델 (몸통) 불러오기
# include_top=False로 머리를 떼어내고, pooling='avg'를 주어 2048차원 벡터로 압축되게 합니다.
base_xception = tf.keras.applications.Xception(
    weights='imagenet', 
    include_top=False, 
    input_shape=(299, 299, 3),
    pooling='avg'
)
print("모델 로드 완료!\n")

# 3. 비교 분석 실행
total_images = 0
match_count = 0
mismatch_list = []

print("=== [ 모델 예측 비교 시작 ] ===")
for img_name in os.listdir(test_dir):
    img_path = os.path.join(test_dir, img_name)
    
    if not img_path.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    total_images += 1

    # ---------------------------------------------------
    # [모델 1] 커스텀 ResNet50 예측 (흑백 처리)
    # ---------------------------------------------------
    img_gray = tf.keras.utils.load_img(img_path, color_mode='grayscale', target_size=resnet_img_size)
    img_gray_arr = tf.keras.utils.img_to_array(img_gray)
    img_gray_arr = tf.image.grayscale_to_rgb(tf.convert_to_tensor(img_gray_arr))
    img_gray_arr = tf.expand_dims(img_gray_arr, 0)
    img_gray_arr = tf.keras.applications.resnet50.preprocess_input(img_gray_arr)
    
    res_score = resnet_model.predict(img_gray_arr, verbose=0)[0][0]
    res_result = class_names[1] if res_score > 0.5 else class_names[0]


    # ---------------------------------------------------
    # [모델 2] Kaggle Xception 예측 (컬러 원본) - 수정됨
    # ---------------------------------------------------
    img_rgb = tf.keras.utils.load_img(img_path, target_size=xception_img_size)
    img_rgb_arr = tf.keras.utils.img_to_array(img_rgb)
    img_rgb_arr = tf.expand_dims(img_rgb_arr, 0)
    img_rgb_arr = tf.keras.applications.xception.preprocess_input(img_rgb_arr)
    
    # ⭐ [수정된 부분] 
    # 1. 베이스 모델을 통과시켜 2048 차원의 특징 추출
    features = base_xception.predict(img_rgb_arr, verbose=0) 
    
    # 2. 추출된 특징(features)을 캐글 분류기 모델에 입력
    xcep_score = xception_classifier.predict(features, verbose=0)[0][0]
    xcep_result = class_names[1] if xcep_score > 0.5 else class_names[0]


    # ---------------------------------------------------
    # 결과 비교 및 출력
    # ---------------------------------------------------
    print(f"▶ 이미지 : {img_name}")
    print(f"  - ResNet50 (흑백) : {res_result.upper():<5} (점수: {res_score:.4f})")
    print(f"  - Xception (컬러) : {xcep_result.upper():<5} (점수: {xcep_score:.4f})")

    if res_result == xcep_result:
        match_count += 1
        print("  => [일치]\n")
    else:
        mismatch_list.append((img_name, res_result, xcep_result))
        print("  => [불일치] ⚠️\n")

# 4. 최종 통계 출력
if total_images > 0:
    print("=== [ 최종 비교 결과 요약 ] ===")
    print(f"총 테스트 이미지 : {total_images}장")
    print(f"두 모델 일치 횟수: {match_count}장")
    print(f"두 모델 불일치 횟수: {total_images - match_count}장")
    print(f"일치율(Agreement) : {(match_count / total_images) * 100:.2f}%\n")

    if mismatch_list:
        print("[판단이 엇갈린 이미지 목록]")
        for name, res_c, xcep_c in mismatch_list:
            print(f" - {name} (ResNet50: {res_c.upper()} / Xception: {xcep_c.upper()})")
else:
    print(f"'{test_dir}' 폴더 내에 테스트할 이미지가 없습니다.")

모델을 불러오는 중입니다...
모델 로드 완료!

=== [ 모델 예측 비교 시작 ] ===
▶ 이미지 : normal0002.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.0084)
  - Xception (컬러) : CLEAN (점수: 0.2118)
  => [일치]

▶ 이미지 : normal0003.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.0679)
  - Xception (컬러) : CLEAN (점수: 0.3253)
  => [일치]

▶ 이미지 : normal0011.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.1030)
  - Xception (컬러) : CLEAN (점수: 0.4447)
  => [일치]

▶ 이미지 : normal0023.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.2507)
  - Xception (컬러) : CLEAN (점수: 0.3809)
  => [일치]

▶ 이미지 : normal0024.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.0126)
  - Xception (컬러) : CLEAN (점수: 0.2916)
  => [일치]

▶ 이미지 : normal0026.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.1041)
  - Xception (컬러) : DIRTY (점수: 0.8489)
  => [불일치] ⚠️

▶ 이미지 : normal0030.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.0019)
  - Xception (컬러) : CLEAN (점수: 0.1661)
  => [일치]

▶ 이미지 : normal0031.jpg
  - ResNet50 (흑백) : CLEAN (점수: 0.2463)
  - Xception (컬러) : CLEAN (점수: 0.4802)
  => [일치]

▶ 이미지 : normal0032.jpg
  - ResNet50 (흑백) : CLEAN